# [SK 07.6.F - AI Foundry Agents using Declarative Spec](https://learn.microsoft.com/en-us/semantic-kernel/frameworks/agent/agent-types/azure-ai-agent?pivots=programming-language-python#declarative-spec)
**Note**: `azure-ai-agents==1.1.0b4` and `azure-ai-projects==1.0.0` are automatically installed by semantic kernel 1.35.0.<br/>

The AzureAIAgent supports instantiation from a YAML declarative specification. The declarative approach allows you to define the agent's properties, instructions, model configuration, tools, and other options in a single, auditable document. This makes agent composition portable and easily managed across environments.<br/>
A minimal YAML declarative spec might look like the following:
```
type: foundry_agent
name: sk_aifoundry_agent-PYTHON-from-specs
instructions: You are a clever agent
description: This agent answers questions
model:
  id: gpt-4o
  options:
    temperature: 0.4
tools:
  - id: LightsPlugin.get_lights
    type: function
  - id: LightsPlugin.change_state
    type: function
```

# Constants and Libraries

Just if needed, log on Azure
```
import os

# Login with tenant ID
os.system("az login --tenant 3ad0b905-34ab-4116-93d9-c1dcc2d35af6 --output none") # --use-device-code

# Set the subscription programmatically
os.system("az account set --subscription eca2eddb-0f0c-4351-a634-52751499eeea")
```

In [1]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
import importlib.metadata

if not load_dotenv("./../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

agent_name = "sk_aifoundry_agent-chocolate-lines"

instructions  = "You are a clever agent that supports the chocolate production lines in this factory."
description   = "This agent answers questions by operators in the chocolate factory." #  supported by Bing to provide grounding context.

project_endpoint = os.environ["AIF_BAS_PROJECT_ENDPOINT"] # AIF_BAS_PROJECT_ENDPOINT or AIF_STD_PROJECT_ENDPOINT
deployment_name =  os.environ["MODEL_DEPLOYMENT_NAME"]
openai_api_version = os.environ["OPENAI_API_VERSION"] # not less than 2025-03-01-preview
openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]

credential = DefaultAzureCredential()

print(f'OpenAI Endpoint: {openai_endpoint}')
print(f'Project Endpoint: {project_endpoint}')
print(f'OpenAI API Version: {openai_api_version}')
print(f"azure-ai-projects library installed version: {importlib.metadata.version("azure-ai-projects")}")
print(f"azure-ai-agents library installed version: {importlib.metadata.version("azure-ai-agents")}")

Environment variables have been loaded ;-)
OpenAI Endpoint: https://mmoaiswc-01.openai.azure.com/
Project Endpoint: https://aif1bassvj36b.services.ai.azure.com/api/projects/aif1basswcprj01
OpenAI API Version: 2025-04-01-preview
azure-ai-projects library installed version: 1.0.0
azure-ai-agents library installed version: 1.1.0b4


# 1. Create AI Foundry `AIProjectClient` using [`AzureAIAgent`](https://learn.microsoft.com/en-us/python/api/semantic-kernel/semantic_kernel.agents.azureaiagent?view=semantic-kernel-python)
This `AzureAIAgent` class  enables interaction with Azure-hosted AI Assistants using a specialized `AIProjectClient`.<br/>
The agent leverages an AzureAIAgentModel configuration and can optionally override default parameters such as temperature, maximum tokens, or instructions.<br/>
Initialize an AzureAIAgent service by providing at minimum an AIProjectClient and an AzureAIAgentModel

In [2]:
from semantic_kernel.agents import AzureAIAgent
os.environ["AZURE_AI_AGENT_ENDPOINT"] = project_endpoint
os.environ["AZURE_AI_AGENT_MODEL_DEPLOYMENT_NAME"] =  deployment_name

project_client = AzureAIAgent.create_client(credential=DefaultAzureCredential())

# 2. Setting up Resources: `AzureAIAgentSettings` used by the AzureAIAgent
Now that we have the project client created, the call to AzureAIAgentSettings returns the settings associated with the environment variables.<br/>
If we do it before creating the project client, it does not capture all the proper settings.

In [3]:
from semantic_kernel.agents import AzureAIAgentSettings

aiagent_settings = AzureAIAgentSettings()
aiagent_settings

AzureAIAgentSettings(env_file_path=None, env_file_encoding='utf-8', model_deployment_name='gpt-4o', endpoint='https://aif1bassvj36b.services.ai.azure.com/api/projects/aif1basswcprj01', agent_id=None, bing_connection_id=None, azure_ai_search_connection_id=None, azure_ai_search_index_name=None, api_version=None)

# Native Plugin

In [4]:
class ProductionLinePlugin:
    from typing import Annotated
    from semantic_kernel.functions import kernel_function

    def __init__(self):
        self.lines = [
            {"id": 0, "name": "Rocher Line", "status": "stopped"},
            {"id": 1, "name": "Mon Chéri Line", "status": "running"},
            {"id": 2, "name": "Kinder Bueno Line", "status": "stopped"},
        ]

    @kernel_function(
        name="get_lines",
        description="Returns the current status of all production lines",
    )
    def get_status(
        self,
    ) -> Annotated[str, "List of production lines and their status"]:
        return str(self.lines)

    @kernel_function(
        name="update_line_status",
        description="Starts or stops a specific production line",
    )
    def update_status(
        self,
        id: int,
        status: str,
    ) -> Annotated[str, "Updated line status"]:
        for line in self.lines:
            if line["id"] == id:
                line["status"] = status
                return str(line)
        return "Line not found"

# Create an AI Foundry Agent

## Define the YAML specification string

In [5]:
spec = f"""
type: foundry_agent
name: {agent_name}
instructions: {instructions}
description: {description}
model:
  id: {aiagent_settings.model_deployment_name}
  options:
    temperature: 0.4
tools:
  - id: ProductionLinePlugin.get_lines
    type: function
  - id: ProductionLinePlugin.update_line_status
    type: function
"""

print(spec)


type: foundry_agent
name: sk_aifoundry_agent-chocolate-lines
instructions: You are a clever agent that supports the chocolate production lines in this factory.
description: This agent answers questions by operators in the chocolate factory.
model:
  id: gpt-4o
  options:
    temperature: 0.4
tools:
  - id: ProductionLinePlugin.get_lines
    type: function
  - id: ProductionLinePlugin.update_line_status
    type: function



## Create the AzureAI Agent from the YAML spec

In [6]:
from semantic_kernel.agents import AgentRegistry

agent: AzureAIAgent = await AgentRegistry.create_from_yaml(
    yaml_str=spec,
    client=project_client,
    plugins=[ProductionLinePlugin()],
    settings=aiagent_settings,
)

# Interacting with an AzureAIAgent
Interaction with the AzureAIAgent is straightforward. The agent maintains the conversation history automatically using a thread.<br/>
The specifics of the Azure AI Agent thread is abstracted away via the AzureAIAgentThread class, which is an implementation of AgentThread.

In [7]:
from semantic_kernel.agents import AzureAIAgentThread
from semantic_kernel.contents import AuthorRole

USER_INPUTS = [
    "Start the Rocher line", 
    "Stop the Kinder Bueno line", 
    "Invert the status of the Mon Chéri line", 
    "Which lines are currently running?",
    "Pause all active lines",
    "Return the status of all lines",
]

thread: AzureAIAgentThread = None

try:
    i=0
    for user_input in USER_INPUTS:
        i+=1
        print(f"Message {i} from {AuthorRole.USER}: '{user_input}'")
        response = await agent.get_response(messages=user_input, thread=thread)
        print(f"Message {i} from {AuthorRole.ASSISTANT}): '{response}'\n")
        thread = response.thread
finally:
    if thread:
        print(f"\nThread <{thread.id}> was created to manage the conversation")

Message 1 from AuthorRole.USER: 'Start the Rocher line'
Message 1 from AuthorRole.ASSISTANT): 'The Rocher Line has been successfully started.'

Message 2 from AuthorRole.USER: 'Stop the Kinder Bueno line'
Message 2 from AuthorRole.ASSISTANT): 'The Kinder Bueno Line has been successfully stopped.'

Message 3 from AuthorRole.USER: 'Invert the status of the Mon Chéri line'
Message 3 from AuthorRole.ASSISTANT): 'The Mon Chéri Line has been successfully stopped, inverting its previous status.'

Message 4 from AuthorRole.USER: 'Which lines are currently running?'
Message 4 from AuthorRole.ASSISTANT): 'Currently, the only production line running is the **Rocher Line**.'

Message 5 from AuthorRole.USER: 'Pause all active lines'
Message 5 from AuthorRole.ASSISTANT): 'All active lines have been paused. The Rocher Line, which was running, has been stopped.'

Message 6 from AuthorRole.USER: 'Return the status of all lines'
Message 6 from AuthorRole.ASSISTANT): 'Here is the current status of all pr

# Teardown

In [8]:
# delete all files
files_to_delete = await project_client.agents.files.list()
files_to_delete_nr = len(files_to_delete.data)

if files_to_delete_nr>0:
    i=0
    print(f"{files_to_delete_nr} files will now be deleted:")
    for f in files_to_delete.data:
        i += 1
        print(f"- File {i} of {files_to_delete_nr}: {f.filename} (id={f.id}) is being deleted...")
        await project_client.agents.files.delete(f.id)
else:
    print("No files to delete")

No files to delete


## Avoiding ***modifying a collection while iterating over it*** for both threads and agents

The code
```
threads_to_delete = project_client.agents.threads.list()
```
returns an async iterator that **lazily** fetches pages of threads.<br/>
But since we're deleting threads as we iterate, the underlying data source is being mutated during iteration. So when the iterator tries to fetch the next page, it hits a missing resource — hence the **ResourceNotFoundError**.<br/><br/>

This is a classic case of *modifying a collection while iterating over it*, which is risky even in synchronous code — and doubly so in async paged APIs.
### The solution
We need to fully materialize the list of threads before deleting anything. That way, the iterator isn’t affected by the deletions

In [9]:
# delete all threads

threads_to_delete = [t async for t in project_client.agents.threads.list()]
i = 0
for t in threads_to_delete:
    i += 1
    print(f"{i} - Thread <{t.id}> is being deleted...")
    await project_client.agents.threads.delete(thread_id=t.id)

1 - Thread <thread_gjtqoiSfT47rO1XWgNdf6FHB> is being deleted...


In [10]:
# delete all agents

agents_to_delete = [a async for a in project_client.agents.list_agents(limit=100)]
i=0
for a in agents_to_delete:
    i += 1
    print(f"{i} - Agent <{a.id}> is being deleted...")
    await project_client.agents.delete_agent(agent_id=a.id)

1 - Agent <asst_M0iU21UJNv2ZGqFI0rlaU9Xd> is being deleted...
